# Real-sky VQA: build the question set from AstroExplorer astrometry

The counterpart of `create_vqa_from_datas.ipynb`, for figures we did **not**
draw. These are real published figures scraped from
[AstroExplorer](https://www.astroexplorer.org) with AVM astrometry embedded in
the JPEG, so the ground truth comes from the astrometric header rather than
from a generator.

Three consequences shape everything below.

**1. Only a subset of the questions can be asked.** For a synthetic figure we
know the colormap, the plot style, the data behind every pixel. Here we know
only where the image points and how big it is. So the colour statistics, the
colormap, the plot style, the distribution and the tick-value questions are all
dropped -- there is no ground truth for them. What survives is the geometry:
field size, pixel scale, and position.

**2. The images cannot be redistributed.** Copyright belongs to the journals.
Only the metadata and the questions are written to
`VQA_realskyimages/`; the images stay local, and
`VQA_realskyimages/fetch_images.py` re-downloads them from AstroExplorer.

**3. The astrometry is embedded in the image file.** Each JPEG carries a 6-8 KB
XMP packet holding the exact AVM coordinates, scale and reference pixel -- i.e.
the answer key. A model reading the file metadata could score perfectly without
looking at the picture. This notebook writes stripped copies and those are what
get sent.

## The epoch trap

The stored coordinates are genuinely **J2000 / FK5** -- verified, not assumed:
across the 122 figures whose title target resolves in SIMBAD, the AVM centre
sits a median **0.025 deg** from the J2000 position (the tightest, AFGL 2591
and DG Tau, agree to 0.0001 deg). If these were B1950 they would be
systematically ~0.6 deg off.

But many of the figures **print B1950 axes**. The Monoceros R2 figure
(1997 AJ 114 209) stores `06h07m46s -06d22m29s`; precess that same point to
B1950 and you get `06h05m20s -06d22m00s`, which is exactly what its axis labels
read. 23% of these papers predate 1990 and the AVM `Spatial.Equinox` field says
`2000.0` for all 156 regardless, so it cannot be used to tell which is which.

Two rules follow, both applied below:

* every positional question says **"J2000 (FK5)"** in its text, so the task is
  well posed even when the printed axes are B1950 (the model must precess);
* **no question asks what the axes say** -- no `epoch` question, no
  `finest tick unit` question. Their ground truth would be whatever the figure
  printed, which the metadata does not record.

The geometry questions (field width, field height, pixel scale) are
frame-independent and carry no such hazard. Their wording is **copied verbatim**
from the synthetic benchmark, so real and synthetic scores are directly
comparable on those three.


In [1]:
# ======================== CONFIG ========================
real_data_dir = '/Users/jnaiman/Dropbox/WASP2026/real_data'

# Metadata + questions go here (public, no images).
out_vqa_dir   = '~/astro_sky_image_vqa/VQA_realskyimages/'

# Stripped images stay with the source data -- NOT in the published dataset.
# This is what actually gets sent to the model.
out_imgs_dir  = real_data_dir + '/images_nometa'

# Figures flagged marginal_correctness=true have astrometry we do not trust
# (wrong position, coordinates copied from another article, or a field too wide
# to map). Their "ground truth" would be wrong, so they are excluded outright.
skip_marginal = True

vqa_prefix = 'rsky_'      # rsky_000001 ... , parallel to the synthetic vqa_NNNNNN

# ---- which questions to ask -------------------------------------------------
# Every entry here has ground truth in the AVM header. Flip any to False to
# drop it; nothing else needs changing.
ASK = {
    # Geometry -- frame-independent, wording identical to the synthetic set,
    # so these three are directly comparable with the synthetic sky figures.
    'field width':   True,
    'field height':  True,
    'pixel scale':   True,

    # Position -- correct ground truth, but the model may have to precess from
    # printed B1950 axes. Every one says "J2000 (FK5)" explicitly.
    'center right ascension': True,
    'center declination':     True,

    # Control: the answer is "image of the sky" for all 156, so this measures
    # nothing about vision -- it is a floor, useful only as a sanity check.
    # Left OFF so it cannot inflate an aggregate accuracy.
    'plot types (list)': False,
}

# Deliberately NOT offered, because the metadata cannot ground them:
#   epoch, finest tick unit  -> depend on what the figure PRINTS (often B1950)
#   colormap, plot style, colour min/max/median/mean, distribution,
#   xtick/ytick values, titles, axis labels -> no ground truth at all

In [2]:
import os, json, glob, re, shutil, collections
import numpy as np
from PIL import Image
from astropy.coordinates import SkyCoord
import astropy.units as u

real_data_dir = os.path.expanduser(real_data_dir)
out_vqa_dir   = os.path.expanduser(out_vqa_dir)
out_imgs_dir  = os.path.expanduser(out_imgs_dir)
out_jsons_dir = os.path.join(out_vqa_dir, 'qa_jsons')
for d in (out_jsons_dir, out_imgs_dir):
    os.makedirs(d, exist_ok=True)

PERSONA = 'You are a helpful assistant that can analyze images.'


def load_records():
    """Every usable AstroExplorer record, in a stable order."""
    recs, skipped = [], []
    for f in sorted(glob.glob(os.path.join(real_data_dir, 'metadata', '*.json'))):
        d = json.load(open(f))
        chk = d.get('astrometry_check') or {}
        if skip_marginal and chk.get('marginal_correctness'):
            skipped.append((d['id'], chk.get('reasons')))
            continue
        recs.append(d)
    return recs, skipped


records, skipped = load_records()
print('usable records : %d' % len(records))
print('skipped (marginal_correctness=true) : %d' % len(skipped))
for i, (rid, why) in enumerate(skipped, 1):
    print('   %d. %s  %s' % (i, rid[-12:], '; '.join(why or []) [:70]))

usable records : 156
skipped (marginal_correctness=true) : 7
   1. 65124518e3fb  title target NGC 7793 is 97.04 deg from image center (field half-diago
   2. 60b4da4e8b5a  title target Kes 79 is 77.92 deg from image center (field half-diagona
   3. ad8b72d160b9  title target HH 25 is 140.81 deg from image center (field half-diagona
   4. 2fe7b8e98e0f  identical reference coordinates shared with figures from 1 other artic
   5. 76d0e4c30ae5  title target NGC 7027 is 156.59 deg from image center (field half-diag
   6. 4a232c9580f1  title target NGC 247 is 118.47 deg from image center (field half-diago
   7. 03283c3b5b9e  field 258 deg across: coordinates may be right, but a single TAN-proje


## Ground truth, straight out of the AVM header

`astrometry_check` already stores the derived footprint (`image_center_radec_deg`,
`field_deg`, `image_size_px`) computed with the WWT conventions -- scale in
arcsec/pixel, reference pixel measured from the bottom-left of the `_hr.jpeg`.
Those are re-derived here rather than trusted blindly, and the two are compared
so a silent disagreement cannot slip through.


In [3]:
def truth_from_record(d, img_path):
    """
    The answerable quantities, in the units the questions ask for.

    Re-derived from `wwt` (scale, ra, dec, x, y) + the real pixel size, then
    cross-checked against the stored `astrometry_check` footprint.
    """
    w = d['wwt']
    scale = float(w['scale'])                    # arcsec / pixel
    ra_c  = float(w['ra'])                       # J2000 FK5, degrees
    dec_c = float(w['dec'])
    xref, yref = float(w['x']), float(w['y'])    # ref pixel, from bottom-left

    with Image.open(img_path) as im:
        nx, ny = im.size

    # Field size.  cos(dec) enters the RA extent, matching the synthetic
    # question's "as a true angle on the sky" wording -- but the ANGULAR width
    # is just nx*scale; the cos factor only matters when converting to a
    # difference in RA coordinate.  Both are reported; the question asks for
    # the true angle, so field_w_arcmin is the answer.
    field_w_arcmin = nx * scale / 60.0
    field_h_arcmin = ny * scale / 60.0

    # Image centre.  (ra, dec) sits at pixel (xref, yref) from the bottom-left;
    # rotation is 180 (north up, east left) for all 163 figures, so the centre
    # is a straight offset.  Guard anyway rather than assume.
    rot = float(w.get('rotation', 180.0))
    if abs(rot - 180.0) > 1e-6:
        raise ValueError('unexpected rotation %.3f for %s' % (rot, d['id']))
    # Proper spherical offset, not a flat cos(dec) approximation: a flat
    # treatment drifts by several arcsec on the wider, higher-declination
    # fields.  This is the same call astrometry_check.py uses, so the two
    # agree exactly rather than approximately.
    cx, cy = nx / 2.0 - xref, ny / 2.0 - yref      # east-left, north-up (rot=180)
    centre = SkyCoord(ra_c * u.deg, dec_c * u.deg, frame='fk5').spherical_offsets_by(
        -cx * scale * u.arcsec, cy * scale * u.arcsec)
    ra_cen, dec_cen = centre.ra.deg % 360.0, centre.dec.deg

    return {'field width': round(field_w_arcmin, 4),
            'field height': round(field_h_arcmin, 4),
            'pixel scale': round(scale, 4),
            'center right ascension': round(ra_cen, 6),
            'center declination': round(dec_cen, 6),
            'plot types (list)': 'image of the sky',
            '_px': [nx, ny]}


# --- cross-check our derivation against the stored footprint ---------------
diffs, scale_list = [], []
for d in records:
    p = os.path.join(real_data_dir, d['image_file'])
    t = truth_from_record(d, p)
    scale_list.append(t['pixel scale'])
    chk = d['astrometry_check']
    fw, fh = chk['field_deg']
    diffs.append((abs(t['field width'] / 60.0 - fw),
                  abs(t['field height'] / 60.0 - fh),
                  abs(t['center right ascension'] - chk['image_center_radec_deg'][0]),
                  abs(t['center declination'] - chk['image_center_radec_deg'][1])))
diffs = np.array(diffs)
# Tolerance in PIXELS, not degrees.  An absolute degree threshold is the wrong
# unit here: pixel scales span 0.0015 to 409 arcsec/px, so a few arcsec is
# invisible on a 39-degree field and enormous on an 18-arcsec one.  Agreement
# to well under one pixel is the meaningful statement.
scales_deg = np.array([t / 3600.0 for t in scale_list])      # deg per pixel
cen_err_px = np.maximum(diffs[:, 2], diffs[:, 3]) / scales_deg

print('agreement with the stored astrometry_check footprint:')
print('   field width   %.2e deg' % diffs[:, 0].max())
print('   field height  %.2e deg' % diffs[:, 1].max())
print('   centre offset %.3f px (max), %.4f px (median)'
      % (cen_err_px.max(), np.median(cen_err_px)))
assert diffs[:, :2].max() < 1e-4, 'field size disagrees with the stored footprint'
assert cen_err_px.max() < 1.0, 'centre disagrees by more than a pixel'
print('OK -- derivation reproduces the stored values to under a pixel')

# The one figure where the stored value rounds the reference pixel to the image
# centre: a 45 arcsec/px, 15x39 deg field.  Ours keeps the sub-pixel offset.
_i = int(np.argmax(cen_err_px))
print('   largest: %s  (%.1f arcsec/px) -- %.2f arcsec, %.5f px'
      % (records[_i]['id'][-12:], scale_list[_i],
         max(diffs[_i, 2], diffs[_i, 3]) * 3600, cen_err_px[_i]))

agreement with the stored astrometry_check footprint:
   field width   5.00e-06 deg
   field height  5.00e-06 deg
   centre offset 0.126 px (max), 0.0000 px (median)
OK -- derivation reproduces the stored values to under a pixel
   largest: 4e085bfbabc2  (45.5 arcsec/px) -- 5.74 arcsec, 0.12616 px


## The questions

Wording for `field width`, `field height` and `pixel scale` is **copied
verbatim** from the synthetic benchmark (lifted out of a released
`VQA_full/qa_jsons` record, not retyped), so those three are directly
comparable between real and synthetic figures.

The two positional questions are new -- the synthetic set asks for *axis
limits*, which assumes a labelled axis. These ask for the **centre of the
image** in an explicitly named frame instead, which is well defined whether or
not the figure draws axes at all.


In [4]:
# Verbatim from the synthetic set -- do not reword, or the comparison breaks.
SYNTHETIC_WORDING = {
 'field width': dict(
    level='Level 2', context='',
    question=('What is the angular width of the sky region shown in this figure, measured '
              'along the right ascension axis, as a true angle on the sky (i.e. including '
              'the cos(declination) factor)? Give the value in arcminutes.'),
    fmt=('Please format the output as a json as {"field width":""} for this figure, '
         'where the value should be a float, expressed in arcminutes.')),
 'field height': dict(
    level='Level 2', context='',
    question=('What is the angular height of the sky region shown in this figure, measured '
              'along the declination axis? Give the value in arcminutes.'),
    fmt=('Please format the output as a json as {"field height":""} for this figure, '
         'where the value should be a float, expressed in arcminutes.')),
 'pixel scale': dict(
    level='Level 3', context='',
    question=('Approximately what angular size does a single pixel of this sky image span? '
              'Give the value in arcseconds.'),
    fmt=('Please format the output as a json as {"pixel scale":""} for this figure, '
         'where the value should be a float, expressed in arcseconds per pixel.')),
 'plot types (list)': dict(
    level='Level 1',
    context='Please choose the plot type from the following list: [contour, image of the sky].',
    question='What is the plot type of the plot in this figure?',
    fmt=('Please format the output as a json as {"plot types":[]}, where the list holds '
         'the plot type of the plot as its only element. ')),
}

# New, real-sky only.  "J2000 (FK5)" is stated explicitly because a figure with
# B1950 axes requires precession -- see the epoch trap at the top.
REALSKY_WORDING = {
 'center right ascension': dict(
    level='Level 3', context='',
    question=('What is the right ascension of the centre of this sky image, in the J2000 '
              '(FK5) frame? Note that the axis labels, if present, may be given in a '
              'different equinox such as B1950; give your answer in J2000. Give the value '
              'in decimal degrees.'),
    fmt=('Please format the output as a json as {"center right ascension":""} for this '
         'figure, where the value should be a float, expressed in decimal degrees J2000.')),
 'center declination': dict(
    level='Level 3', context='',
    question=('What is the declination of the centre of this sky image, in the J2000 (FK5) '
              'frame? Note that the axis labels, if present, may be given in a different '
              'equinox such as B1950; give your answer in J2000. Give the value in decimal '
              'degrees.'),
    fmt=('Please format the output as a json as {"center declination":""} for this figure, '
         'where the value should be a float, expressed in decimal degrees J2000.')),
}

WORDING = dict(SYNTHETIC_WORDING, **REALSKY_WORDING)
ACTIVE = [k for k, v in ASK.items() if v]
print('asking %d questions per figure:' % len(ACTIVE))
for k in ACTIVE:
    print('   %-26s %s' % (k, WORDING[k]['level']))

asking 5 questions per figure:
   field width                Level 2
   field height               Level 2
   pixel scale                Level 3
   center right ascension     Level 3
   center declination         Level 3


In [5]:
def build_vqa(truth):
    """
    VQA block in the same shape the run notebooks expect:
    VQA[level]['Figure-level questions'][name] -> entry.

    Everything sits at figure level: a real published figure is one image, so
    there is no per-panel axis to key on.  Entry fields match the synthetic set
    exactly (Q, A, persona, context, question, format), so the same response
    parser works on both.
    """
    vqa = {}
    for name in ACTIVE:
        w = WORDING[name]
        ans = truth[name]
        if name == 'plot types (list)':
            a = {'plot types': [ans]}
        else:
            a = {name: ans}
        Q = ' '.join(x for x in (PERSONA, w['context'], w['question'], w['fmt']) if x)
        vqa.setdefault(w['level'], {}).setdefault('Figure-level questions', {})[name] = {
            'Q': Q, 'A': a, 'persona': PERSONA, 'context': w['context'],
            'question': w['question'], 'format': w['fmt'],
        }
    return vqa


def public_metadata(d, truth):
    """What may be published: provenance + astrometry, never the image."""
    return {
        'id': d['id'],
        'details_url': d.get('details_url'),
        'hr_url': d.get('hr_url'),                 # how to re-fetch the image
        'source_image_file': os.path.basename(d['image_file']),
        'title': d.get('title'),
        'authors': d.get('authors'),
        'citation': d.get('citation'),
        'doi': d.get('doi'),
        'ads_url': d.get('ads_url'),
        'copyright': d.get('copyright'),
        'caption': d.get('caption'),
        'avm': d.get('avm'),
        'wwt': d.get('wwt'),
        'astrometry_check': d.get('astrometry_check'),
        'image_size_px': truth['_px'],
    }

## Strip the metadata from the images

Each JPEG carries the AVM astrometry in an XMP packet -- the answer key, inside
the file we are about to send. `Image.save()` on the pixel data alone writes a
clean file: no XMP, no EXIF, no JFIF density.

Re-encoding costs a JPEG generation, so quality is set high (95) and subsampling
disabled. The alternative -- copying the file and surgically deleting the APP1
segment -- would avoid that, but leaves whatever else is in the container; a
clean re-encode is easier to be sure about. The check below reopens every
output and asserts no metadata survived.


In [6]:
def strip_metadata(src, dst, quality=95):
    """Re-encode pixels only -- no XMP, EXIF, ICC or JFIF density carried over."""
    with Image.open(src) as im:
        im = im.convert('RGB')
        data = list(im.getdata())
        clean = Image.new('RGB', im.size)
        clean.putdata(data)
        clean.save(dst, 'JPEG', quality=quality, subsampling=0, optimize=True)


METADATA_KEYS = ('xmp', 'exif', 'icc_profile', 'photoshop', 'adobe', 'comment')

def metadata_bytes(path):
    with Image.open(path) as im:
        n = sum(len(im.info[k]) for k in METADATA_KEYS
                if k in im.info and hasattr(im.info[k], '__len__'))
    raw = open(path, 'rb').read()
    # belt and braces: the literal AVM namespace must not appear anywhere
    n += raw.count(b'avm:') + raw.count(b'<x:xmpmeta')
    return n

In [7]:
# ======================== BUILD ========================
rows = []
for i, d in enumerate(records, 1):
    vqa_id   = '%s%06d' % (vqa_prefix, i)
    src      = os.path.join(real_data_dir, d['image_file'])
    ext      = os.path.splitext(src)[1].lower()
    dst      = os.path.join(out_imgs_dir, vqa_id + '.jpeg')

    truth = truth_from_record(d, src)
    strip_metadata(src, dst)

    record = {
        'source': public_metadata(d, truth),
        'image': {'vqa_id': vqa_id,
                  'stripped_file': os.path.basename(dst),
                  'width': truth['_px'][0], 'height': truth['_px'][1]},
        'VQA': build_vqa(truth),
        'vqa_id': vqa_id,
    }

    # Double-encoded exactly like VQA_full/qa_jsons, so the run notebooks'
    # get_img_json_pair() (which does json.loads(json.load(f))) works unchanged.
    with open(os.path.join(out_jsons_dir, vqa_id + '_qa.json'), 'w') as f:
        json.dump(json.dumps(record), f)

    rows.append({'vqa_id': vqa_id, 'id': d['id'],
                 'title': (d.get('title') or '')[:60],
                 'field_w_arcmin': truth['field width'],
                 'pixel_scale_arcsec': truth['pixel scale'],
                 'px': 'x'.join(map(str, truth['_px'])),
                 'src_meta_bytes': metadata_bytes(src),
                 'out_meta_bytes': metadata_bytes(dst)})

print('wrote %d qa jsons -> %s' % (len(rows), out_jsons_dir))
print('wrote %d stripped images -> %s' % (len(rows), out_imgs_dir))

/var/folders/1c/xcwp8_px05xg1_3xd9nn9qhm0000gn/T/ipykernel_54356/1950783301.py:5: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  data = list(im.getdata())


wrote 156 qa jsons -> /Users/jnaiman/astro_sky_image_vqa/VQA_realskyimages/qa_jsons
wrote 156 stripped images -> /Users/jnaiman/Dropbox/WASP2026/real_data/images_nometa


In [8]:
# ======================== VERIFY ========================
import pandas as pd
df = pd.DataFrame(rows)

leaks = df[df['out_meta_bytes'] > 0]
print('METADATA STRIP')
print('   source images carrying metadata : %d of %d (median %d bytes)'
      % ((df['src_meta_bytes'] > 0).sum(), len(df), int(df['src_meta_bytes'].median())))
print('   stripped images with ANY residue: %d' % len(leaks))
assert leaks.empty, 'metadata survived in:\n%s' % leaks[['vqa_id', 'out_meta_bytes']]

# every json must re-read and carry the expected questions
n_q = collections.Counter()
for p in sorted(glob.glob(os.path.join(out_jsons_dir, '*_qa.json'))):
    rec = json.loads(json.load(open(p)))
    got = [n for lv in rec['VQA'].values() for kind in lv.values() for n in kind]
    n_q[len(got)] += 1
    assert sorted(got) == sorted(ACTIVE), (p, got)
print('\nQA JSONS')
print('   re-read OK, questions per figure:', dict(n_q))
print('   total questions: %d' % (len(df) * len(ACTIVE)))

print('\nFIELD SIZES (arcmin)')
print(df['field_w_arcmin'].describe()[['min', '50%', 'max']].to_string())
print('\nPIXEL SCALE (arcsec/px)')
print(df['pixel_scale_arcsec'].describe()[['min', '50%', 'max']].to_string())
df.head(8)

METADATA STRIP
   source images carrying metadata : 156 of 156 (median 7348 bytes)
   stripped images with ANY residue: 0

QA JSONS
   re-read OK, questions per figure: {5: 156}
   total questions: 780

FIELD SIZES (arcmin)
min       0.26130
50%      13.55495
max    1636.15700

PIXEL SCALE (arcsec/px)
min     0.0116
50%     0.4133
max    45.4834


,vqa_id,id,title,field_w_arcmin,pixel_scale_arcsec,px,src_meta_bytes,out_meta_bytes
0,rsky_000001,308158864287067800170087_0000209.000,Properties of the Monoceros R2 Stellar Cluster,18.8924,1.1381,996x1135,8024,0
1,rsky_000002,308158864287067800170087_0000915.000,Close galaxy pairs in low and medium density r...,8.5127,0.4925,1037x741,6589,0
2,rsky_000003,308158864287067800170087_0000918.000,Close galaxy pairs in low and medium density r...,9.0497,0.7163,758x975,6589,0
3,rsky_000004,308158864287067800170087_0000924.000,Close galaxy pairs in low and medium density r...,150.5867,8.5399,1058x1002,6588,0
4,rsky_000005,308158864287067800170087_0001420.000,Structural Components of NGC 5850.,8.0598,0.2128,2272x2973,7338,0
5,rsky_000006,308158864287067800170087_0001421.000,Structural Components of NGC 5850.,8.0636,0.2129,2272x2956,7341,0
6,rsky_000007,308158864287067800170087_0001430.000,NGC 5291: Implications for the Formation of Dw...,19.3091,0.7455,1554x1934,8460,0
7,rsky_000008,308158864287067800170087_0001431.000,NGC 5291: Implications for the Formation of Dw...,19.4980,0.7485,1563x1925,8453,0


## The image fetcher and the dataset README

`VQA_realskyimages/` holds metadata and questions only. These two files are what
let someone else reconstruct the image set.


In [9]:
# The fetcher shipped with the dataset.  Written with "#" comments rather than
# a docstring so there is no nested-quote hazard in this template.
FETCHER = '''#!/usr/bin/env python3
# Re-download the AstroExplorer figures this VQA set describes, and rebuild the
# stripped copies that were actually sent to the models.
#
# The images are NOT redistributed with this dataset -- copyright belongs to the
# journals (see source.copyright in each qa json).  Every record keeps the
# AstroExplorer details URL and the direct hi-res URL, so the set can be rebuilt.
#
#     python3 fetch_images.py                 -> ./images and ./images_nometa
#     python3 fetch_images.py --out DIR
#
# Needs pillow for the metadata strip; nothing else beyond the stdlib.
#
# Note on the strip: each JPEG as served carries the AVM astrometry in an XMP
# packet -- that is the answer key to most of these questions.  images_nometa/
# holds re-encoded, metadata-free copies.  Send those to a model, never images/.
import argparse, json, glob, os, sys, time, urllib.request

AE_RATE_LIMIT_S = 1.0


def fetch(url, tries=4):
    for a in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read()
        except Exception:
            if a == tries - 1:
                raise
            time.sleep(2 * (a + 1))


def strip(src, dst):
    from PIL import Image
    with Image.open(src) as im:
        im = im.convert("RGB")
        clean = Image.new("RGB", im.size)
        clean.putdata(list(im.getdata()))
        clean.save(dst, "JPEG", quality=95, subsampling=0, optimize=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=".", help="where to write images/ and images_nometa/")
    ap.add_argument("--jsons", default="qa_jsons", help="directory of *_qa.json")
    a = ap.parse_args()
    raw = os.path.join(a.out, "images")
    clean = os.path.join(a.out, "images_nometa")
    for d in (raw, clean):
        os.makedirs(d, exist_ok=True)

    paths = sorted(glob.glob(os.path.join(a.jsons, "*_qa.json")))
    if not paths:
        sys.exit("no qa jsons found in " + a.jsons)
    print(str(len(paths)) + " figures to fetch")

    ok = skipped = failed = 0
    for i, p in enumerate(paths, 1):
        rec = json.loads(json.load(open(p)))
        vid = rec["vqa_id"]
        url = rec["source"]["hr_url"]
        dst_raw = os.path.join(raw, vid + os.path.splitext(url)[1])
        dst_cln = os.path.join(clean, vid + ".jpeg")
        if os.path.exists(dst_cln):
            skipped += 1
            continue
        try:
            if not os.path.exists(dst_raw):
                open(dst_raw, "wb").write(fetch(url))
                time.sleep(AE_RATE_LIMIT_S)
            strip(dst_raw, dst_cln)
            ok += 1
        except Exception as e:
            print("  [FAIL] " + vid + "  " + url + ": " + str(e))
            failed += 1
        if i % 25 == 0:
            print("  " + str(i) + "/" + str(len(paths)))

    print("done: %d fetched, %d already present, %d failed" % (ok, skipped, failed))
    print("send images_nometa/ to models -- images/ still carries the AVM answer key")


if __name__ == "__main__":
    main()
'''

path = os.path.join(out_vqa_dir, 'fetch_images.py')
with open(path, 'w') as f:
    f.write(FETCHER)
os.chmod(path, 0o755)
print('wrote', path, '(%d lines)' % FETCHER.count(chr(10)))


wrote /Users/jnaiman/astro_sky_image_vqa/VQA_realskyimages/fetch_images.py (84 lines)


In [10]:
# Dataset README.  Uses .replace() rather than % formatting so literal percent
# signs in the prose need no escaping.
_qlist = '\n'.join(
    '* **`' + k + '`** (' + WORDING[k]['level'] + ') -- '
    + WORDING[k]['question'].split('?')[0][:115] + '?'
    for k in ACTIVE)

README = """# VQA_realskyimages

Questions and ground truth for **{N} real published astronomical figures**,
scraped from [AstroExplorer](https://www.astroexplorer.org) with AVM astrometry
embedded in each JPEG.

The companion to `VQA_full/`, which holds synthetic figures. The difference that
matters: these were not drawn by us, so the only ground truth is the astrometric
header -- where the image points, how wide it is, and its pixel scale.

## No images here

**Copyright belongs to the journals** (`source.copyright` in each record), so
only metadata and questions are distributed. Rebuild the image set with:

    python3 fetch_images.py

That writes `images/` (as served, AVM metadata intact) and `images_nometa/`
(re-encoded, metadata stripped). **Send `images_nometa/` to a model** -- the
served files carry the answer key in an XMP packet.

## Contents

    qa_jsons/rsky_NNNNNN_qa.json   {N} records: provenance + astrometry + questions
    fetch_images.py                re-downloads and re-strips the images

Each json is a JSON-encoded *string* (two decodes, as in `VQA_full`):

    rec = json.loads(json.load(open(path)))
    rec.keys()   # ['source', 'image', 'VQA', 'vqa_id']

| key | contents |
|---|---|
| `source` | AstroExplorer id, details/hi-res URLs, title, authors, citation, DOI, ADS link, copyright, the full `avm` block, the `wwt` parameters, and `astrometry_check` |
| `image` | `vqa_id`, stripped filename, pixel width/height |
| `VQA` | `VQA[level]['Figure-level questions'][name]` -> entry, same entry fields as `VQA_full` |
| `vqa_id` | `rsky_NNNNNN` |

## Selection

163 figures were scraped; **7 are excluded** because `astrometry_check` flags
`marginal_correctness` (wrong position, coordinates copied from a neighbouring
article, or a field too wide to map with one TAN projection). The remaining
**{N}** are here.

## The questions, and why these

{QLIST}

Dropped, because the metadata cannot ground them: colormap, plot style, colour
min/max/median/mean, the underlying distribution, tick values, titles and axis
labels -- none of these are recorded for a figure we did not draw.

Also dropped, for a subtler reason: **`epoch` and `finest tick unit`**. Those
ask what the figure *prints*. The AVM `Spatial.Equinox` field reads `2000.0` for
all {N} records, but that describes the astrometric solution, not the axis
labels -- the Monoceros R2 figure (1997 AJ 114 209) stores J2000
`06h07m46s -06d22m29s` while its axes are labelled `alpha[1950]`. 23% of these
papers predate 1990. There is no record of what each figure printed, so these
questions have no ground truth.

## The frame is J2000, and it was checked

The stored coordinates are **J2000 / FK5**. Verified rather than assumed: across
the 122 figures whose title target resolves in SIMBAD, the AVM centre lies a
median **0.025 deg** from the J2000 position, the closest agreeing to 0.0001
deg. B1950 values would be systematically ~0.6 deg off.

Because some figures nevertheless *print* B1950 axes, both positional questions
state "J2000 (FK5)" explicitly and warn that the labels may use another equinox.
Answering them from a B1950-labelled figure requires precessing.

The three geometry questions are frame-independent and carry no such hazard.
Their wording is **identical to the synthetic benchmark's**, so scores on
`field width`, `field height` and `pixel scale` are directly comparable between
real and synthetic figures.

## TODO: the positional questions have no synthetic counterpart

`center right ascension` and `center declination` cannot currently be compared
against anything in `VQA_full`. The synthetic sky figures ask for **axis
limits** (`minimum/maximum right ascension axis limit`, and the declination
pair), which presumes a drawn, labelled axis -- fine for a figure we rendered,
but not something every published figure has. So the real-sky set asks for the
**centre of the image** instead, and the two are not the same question.

The consequence: the three geometry questions (`field width`, `field height`,
`pixel scale`) are worded identically in both sets and can be read side by
side, but the positional result here -- median 0.45 deg, 70% within 1 deg on
156 figures -- is a standalone number with no synthetic baseline to sit
against. There is no way to say whether that is good *relative to* performance
on figures where we know the answer is cleanly readable.

Worth closing later, if the comparison turns out to matter. Either direction
would work:

* **Add centre questions to the synthetic set.** The generator knows each
  figure's central RA/Dec exactly, so `center right ascension` /
  `center declination` could be emitted for the synthetic sky panels using
  the wording already used here. Cheapest option, and it makes the two sets
  line up on all five questions. Costs a re-run of the synthetic sky figures.
* **Add axis-limit questions to the real set.** Harder and probably not worth
  it: the AVM header gives the image footprint, not what the figure's axes
  are annotated to, and many of these figures have no usable axis at all.

The first is the better route. Note that if the synthetic centre questions are
added, they should carry the same explicit "J2000 (FK5)" framing used here, or
the two still will not be measuring quite the same thing.

## Licence

The questions, ground truth and metadata in this directory are released under
the same terms as the rest of this dataset. The **figures themselves are not
included and are not ours to license** -- each `source.copyright` names the
journal that holds them.
""".replace('{N}', str(len(records))).replace('{QLIST}', _qlist)

path = os.path.join(out_vqa_dir, 'README.md')
with open(path, 'w') as f:
    f.write(README)
print('wrote', path)
print()
print(README[:900])


wrote /Users/jnaiman/astro_sky_image_vqa/VQA_realskyimages/README.md

# VQA_realskyimages

Questions and ground truth for **156 real published astronomical figures**,
scraped from [AstroExplorer](https://www.astroexplorer.org) with AVM astrometry
embedded in each JPEG.

The companion to `VQA_full/`, which holds synthetic figures. The difference that
matters: these were not drawn by us, so the only ground truth is the astrometric
header -- where the image points, how wide it is, and its pixel scale.

## No images here

**Copyright belongs to the journals** (`source.copyright` in each record), so
only metadata and questions are distributed. Rebuild the image set with:

    python3 fetch_images.py

That writes `images/` (as served, AVM metadata intact) and `images_nometa/`
(re-encoded, metadata stripped). **Send `images_nometa/` to a model** -- the
served files carry the answer key in an XMP packet.

## Contents

    qa_jsons/rsky_NNNNNN_qa.json   156 records
